# Indian Job Market Intelligence — Combined Notebook

This notebook combines the uploaded Python modules into a single `.ipynb` file. Each original file is kept as a separate section, preserving its source code.

## data_loader(1).py

In [ ]:
"""
data_loader.py — Responsible for reading the CSV from disk and returning
a raw DataFrame. Keeps I/O separate from cleaning logic.

NOTE: No Streamlit imports here. Caching (@st.cache_data) is applied
by callers (app.py, pages) that run inside the Streamlit runtime.
"""

import os
import pandas as pd

# Default path relative to the project root
DEFAULT_DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "indian_job_market.csv")


def load_raw_data(filepath: str = DEFAULT_DATA_PATH) -> pd.DataFrame:
    """
    Load the raw CSV dataset and return as a DataFrame.
    Raises FileNotFoundError with a friendly message if the file is missing.
    Pure function — no Streamlit dependency.
    """
    filepath = os.path.abspath(filepath)
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"Dataset not found at: {filepath}\n"
            "Please place 'indian_job_market.csv' inside the 'data/' folder."
        )
    df = pd.read_csv(filepath, low_memory=False)
    return df


def get_dataset_info(df: pd.DataFrame) -> dict:
    """Return a summary dict about the raw dataset for display purposes."""
    return {
        "rows": df.shape[0],
        "columns": df.shape[1],
        "column_names": df.columns.tolist(),
        "null_counts": df.isnull().sum().to_dict(),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "duplicates": df.duplicated().sum(),
    }


## data_cleaning(2).py

In [ ]:
"""
data_cleaning.py — All preprocessing and cleaning logic.
Every function is pure: takes a DataFrame, returns a cleaned DataFrame.
"""

import re
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Salary cleaning
# ---------------------------------------------------------------------------

def _clean_salary_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace 0 salary values with NaN (0 means 'Not disclosed' in this dataset).
    Convert INR salaries in USD to INR where currency == USD.
    Derive average_salary = (minimumSalary + maximumSalary) / 2.
    """
    df = df.copy()

    # Convert USD salaries to INR (approximate: 1 USD ≈ 83 INR)
    usd_mask = df["currency"] == "USD"
    for col in ["minimumSalary", "maximumSalary"]:
        df.loc[usd_mask, col] = df.loc[usd_mask, col] * 83

    # Zero salary means undisclosed
    for col in ["minimumSalary", "maximumSalary"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[df[col] == 0, col] = np.nan

    # Sanity check: min should not exceed max
    swap_mask = df["minimumSalary"] > df["maximumSalary"]
    df.loc[swap_mask, ["minimumSalary", "maximumSalary"]] = (
        df.loc[swap_mask, ["maximumSalary", "minimumSalary"]].values
    )

    # Derived average salary
    df["average_salary"] = (df["minimumSalary"] + df["maximumSalary"]) / 2

    # Cap extreme outliers at 99th percentile (keeps realistic values)
    cap = df["average_salary"].quantile(0.99)
    df.loc[df["average_salary"] > cap, "average_salary"] = np.nan

    return df


# ---------------------------------------------------------------------------
# Experience cleaning
# ---------------------------------------------------------------------------

def _clean_experience_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure numeric experience columns are valid.
    Replace 0-0 (both zero) with NaN.
    Derive average_experience.
    """
    df = df.copy()
    for col in ["minimumExperience", "maximumExperience"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # If both are 0 it likely means undisclosed
    both_zero = (df["minimumExperience"] == 0) & (df["maximumExperience"] == 0)
    df.loc[both_zero, ["minimumExperience", "maximumExperience"]] = np.nan

    df["average_experience"] = (
        df["minimumExperience"].fillna(0) + df["maximumExperience"].fillna(0)
    ) / 2

    return df


# ---------------------------------------------------------------------------
# Location cleaning
# ---------------------------------------------------------------------------

def _clean_location(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract the primary city from a multi-city location string.
    E.g. 'Noida, Greater Noida' → 'Noida'
         'Kolkata(Chinar Park)' → 'Kolkata'
    """
    df = df.copy()
    df["location"] = df["location"].astype(str).str.strip()

    def extract_city(loc: str) -> str:
        # Take first city before comma
        city = loc.split(",")[0].strip()
        # Remove parenthetical sub-areas
        city = re.sub(r"\(.*?\)", "", city).strip()
        return city if city else "Unknown"

    df["primary_location"] = df["location"].apply(extract_city)
    return df


# ---------------------------------------------------------------------------
# Title / company cleaning
# ---------------------------------------------------------------------------

def _clean_titles(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise job titles: strip extra spaces, fix common casing."""
    df = df.copy()
    df["title"] = df["title"].astype(str).str.strip()
    df["title"] = df["title"].str.replace(r"\s+", " ", regex=True)
    return df


def _clean_company_names(df: pd.DataFrame) -> pd.DataFrame:
    """Strip and normalise company names."""
    df = df.copy()
    df["companyName"] = df["companyName"].astype(str).str.strip()
    df.loc[df["companyName"].isin(["nan", "None", ""]), "companyName"] = "Unknown"
    return df


# ---------------------------------------------------------------------------
# Date / upload time
# ---------------------------------------------------------------------------

def _parse_job_uploaded(df: pd.DataFrame) -> pd.DataFrame:
    """
    'jobUploaded' contains relative strings like '4 Days Ago', 'Just Now'.
    Convert to approximate numeric days_ago for time-series approximation.
    """
    df = df.copy()

    def to_days_ago(s: str) -> int:
        s = str(s).strip().lower()
        if s in ("just now", "today", "few hours ago"):
            return 0
        m = re.search(r"(\d+)\s*day", s)
        if m:
            return int(m.group(1))
        if "1 day" in s:
            return 1
        # "Starts in …" — treat as future / unknown
        return np.nan

    df["days_ago"] = df["jobUploaded"].apply(to_days_ago)
    return df


# ---------------------------------------------------------------------------
# Ratings
# ---------------------------------------------------------------------------

def _clean_ratings(df: pd.DataFrame) -> pd.DataFrame:
    """Keep AggregateRating as float; leave NaN where unavailable."""
    df = df.copy()
    df["AggregateRating"] = pd.to_numeric(df["AggregateRating"], errors="coerce")
    df["ReviewsCount"] = pd.to_numeric(df["ReviewsCount"], errors="coerce")
    return df


# ---------------------------------------------------------------------------
# Master cleaner
# ---------------------------------------------------------------------------

def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run all cleaning steps in order and return a cleaned DataFrame.
    """
    df = df.copy()

    # 1. Drop exact duplicates
    df = df.drop_duplicates()

    # 2. Column-level cleaning
    df = _clean_salary_column(df)
    df = _clean_experience_columns(df)
    df = _clean_location(df)
    df = _clean_titles(df)
    df = _clean_company_names(df)
    df = _parse_job_uploaded(df)
    df = _clean_ratings(df)

    # 3. Drop rows where both title and companyName are missing
    df = df.dropna(subset=["title"])

    return df.reset_index(drop=True)


## feature_engineering(1).py

In [ ]:
"""
feature_engineering.py — Derives new columns used across all pages.
All functions are pure transformers.
"""

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Experience buckets
# ---------------------------------------------------------------------------

EXP_BINS = [-0.1, 0, 2, 5, 8, 100]
EXP_LABELS = ["Fresher (0 Yrs)", "0–2 Yrs", "2–5 Yrs", "5–8 Yrs", "8+ Yrs"]


def add_experience_group(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create 'experience_group' from minimumExperience.
    Fresher = minimumExperience == 0
    """
    df = df.copy()
    df["experience_group"] = pd.cut(
        df["minimumExperience"].fillna(-1),
        bins=EXP_BINS,
        labels=EXP_LABELS,
        right=True,
    ).astype(str)
    df.loc[df["experience_group"] == "nan", "experience_group"] = "Unknown"
    return df


# ---------------------------------------------------------------------------
# Salary bucket (in Lakhs PA for readability)
# ---------------------------------------------------------------------------

def add_salary_lpa(df: pd.DataFrame) -> pd.DataFrame:
    """Convert rupee salary columns to Lakhs Per Annum (LPA) for display."""
    df = df.copy()
    for col in ["minimumSalary", "maximumSalary", "average_salary"]:
        if col in df.columns:
            df[f"{col}_lpa"] = (df[col] / 100_000).round(2)
    return df


# ---------------------------------------------------------------------------
# Fresher flag
# ---------------------------------------------------------------------------

def add_fresher_flag(df: pd.DataFrame) -> pd.DataFrame:
    """Mark rows suitable for freshers (minimumExperience == 0)."""
    df = df.copy()
    df["is_fresher"] = (
        df["minimumExperience"].fillna(99) == 0
    )
    return df


# ---------------------------------------------------------------------------
# Master feature engineer
# ---------------------------------------------------------------------------

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Run all feature engineering steps."""
    df = add_experience_group(df)
    df = add_salary_lpa(df)
    df = add_fresher_flag(df)
    return df


## skill_extraction(1).py

In [ ]:
"""
skill_extraction.py — Parses the comma-separated tagsAndSkills column and
maps raw skill strings to canonical technology names.
"""

import re
from collections import Counter
from itertools import combinations

import pandas as pd

# ---------------------------------------------------------------------------
# Canonical skill map  (raw substring → display name)
# Longer / more-specific patterns should appear before shorter ones.
# ---------------------------------------------------------------------------

SKILL_MAP = {
    # Data & Analytics
    "machine learning": "Machine Learning",
    "deep learning": "Deep Learning",
    "natural language": "NLP",
    "nlp": "NLP",
    "data analysis": "Data Analysis",
    "data analytics": "Data Analytics",
    "data science": "Data Science",
    "data engineering": "Data Engineering",
    "data visualization": "Data Visualization",
    "data visualisation": "Data Visualization",
    "business intelligence": "Business Intelligence",
    "business analytics": "Business Analytics",
    "statistical analysis": "Statistical Analysis",
    "statistics": "Statistics",
    "predictive": "Predictive Modeling",
    # Languages
    "python": "Python",
    "r programming": "R",
    " r ": "R",
    "java": "Java",
    "javascript": "JavaScript",
    "typescript": "TypeScript",
    "c++": "C++",
    "c#": "C#",
    "scala": "Scala",
    "go lang": "Go",
    "golang": "Go",
    "rust": "Rust",
    "kotlin": "Kotlin",
    "swift": "Swift",
    "php": "PHP",
    "ruby": "Ruby",
    "perl": "Perl",
    "shell": "Shell/Bash",
    "bash": "Shell/Bash",
    # Databases
    "sql": "SQL",
    "mysql": "MySQL",
    "postgresql": "PostgreSQL",
    "oracle": "Oracle DB",
    "mongodb": "MongoDB",
    "nosql": "NoSQL",
    "cassandra": "Cassandra",
    "redis": "Redis",
    "elasticsearch": "Elasticsearch",
    "hadoop": "Hadoop",
    "hive": "Hive",
    "spark": "Apache Spark",
    # Cloud
    "aws": "AWS",
    "amazon web": "AWS",
    "azure": "Azure",
    "gcp": "GCP",
    "google cloud": "GCP",
    # BI Tools
    "power bi": "Power BI",
    "tableau": "Tableau",
    "looker": "Looker",
    "qlik": "Qlik",
    "excel": "Excel",
    "ms excel": "Excel",
    "microsoft excel": "Excel",
    # ML Libraries
    "tensorflow": "TensorFlow",
    "pytorch": "PyTorch",
    "scikit": "Scikit-learn",
    "sklearn": "Scikit-learn",
    "keras": "Keras",
    "xgboost": "XGBoost",
    "pandas": "Pandas",
    "numpy": "NumPy",
    "matplotlib": "Matplotlib",
    "seaborn": "Seaborn",
    "plotly": "Plotly",
    # DevOps / Infra
    "docker": "Docker",
    "kubernetes": "Kubernetes",
    "jenkins": "Jenkins",
    "git": "Git",
    "ci/cd": "CI/CD",
    "terraform": "Terraform",
    "ansible": "Ansible",
    # Web / APIs
    "react": "React",
    "angular": "Angular",
    "vue": "Vue.js",
    "node.js": "Node.js",
    "nodejs": "Node.js",
    "django": "Django",
    "flask": "Flask",
    "fastapi": "FastAPI",
    "spring": "Spring",
    "rest api": "REST API",
    "restful": "REST API",
    "graphql": "GraphQL",
    "microservices": "Microservices",
    # General / Soft
    "communication": "Communication",
    "leadership": "Leadership",
    "project management": "Project Management",
    "agile": "Agile",
    "scrum": "Scrum",
    "teamwork": "Teamwork",
    "problem solving": "Problem Solving",
    "analytical": "Analytical Skills",
    "ms office": "MS Office",
    "microsoft office": "MS Office",
    "salesforce": "Salesforce",
    "erp": "ERP",
    "sap": "SAP",
}


def _normalize(text: str) -> str:
    return text.lower().strip()


def extract_skills_from_row(raw: str) -> list[str]:
    """
    Parse a comma-separated tagsAndSkills cell and return a list of
    canonical skill names (deduplicated, in insertion order).
    """
    if not isinstance(raw, str) or raw.strip() == "":
        return []

    tokens = [t.strip() for t in raw.split(",")]
    found = []
    seen = set()
    for token in tokens:
        norm = _normalize(token)
        for pattern, canonical in SKILL_MAP.items():
            if pattern in norm and canonical not in seen:
                found.append(canonical)
                seen.add(canonical)
                break
    return found


def add_skills_list(df: pd.DataFrame) -> pd.DataFrame:
    """Add a 'skills_list' column (list of canonical skills per row)."""
    df = df.copy()
    df["skills_list"] = df["tagsAndSkills"].apply(extract_skills_from_row)
    return df


def get_top_skills(df: pd.DataFrame, n: int = 30) -> pd.DataFrame:
    """
    Explode skills_list and return the top-n most frequent skills with counts.
    """
    all_skills = [skill for skills in df["skills_list"] for skill in skills]
    counts = Counter(all_skills)
    top = pd.DataFrame(counts.most_common(n), columns=["skill", "count"])
    return top


def get_skill_by_column(df: pd.DataFrame, col: str, n_skills: int = 15) -> pd.DataFrame:
    """
    Return a pivot-style DataFrame: skill vs. category (col values) with counts.
    Useful for skill vs. job role, skill vs. experience, etc.
    """
    rows = []
    for _, row in df[[col, "skills_list"]].dropna(subset=[col]).iterrows():
        for skill in row["skills_list"]:
            rows.append({"category": row[col], "skill": skill})
    if not rows:
        return pd.DataFrame(columns=["category", "skill", "count"])
    result = (
        pd.DataFrame(rows)
        .groupby(["category", "skill"])
        .size()
        .reset_index(name="count")
    )
    # Keep only top skills overall for readability
    top_skills = get_top_skills(df, n_skills)["skill"].tolist()
    result = result[result["skill"].isin(top_skills)]
    return result


def get_skill_cooccurrence(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    """
    Return a co-occurrence matrix for the top-n skills.
    """
    top_skills = set(get_top_skills(df, top_n)["skill"].tolist())
    co: dict = {}
    for skills in df["skills_list"]:
        filtered = [s for s in skills if s in top_skills]
        for a, b in combinations(sorted(filtered), 2):
            key = (a, b)
            co[key] = co.get(key, 0) + 1
    if not co:
        return pd.DataFrame(columns=["skill_a", "skill_b", "count"])
    rows = [{"skill_a": a, "skill_b": b, "count": c} for (a, b), c in co.items()]
    return pd.DataFrame(rows).sort_values("count", ascending=False)


## analysis(1).py

In [ ]:
"""
analysis.py — Reusable aggregation helpers used by the page modules.
All functions accept a (possibly pre-filtered) DataFrame and return
a results DataFrame ready for plotting.
"""

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------------

def top_n_by_count(df: pd.DataFrame, col: str, n: int = 10) -> pd.DataFrame:
    """Return top-n values of col by frequency, as a DataFrame."""
    return (
        df[col]
        .value_counts()
        .head(n)
        .reset_index()
        .rename(columns={"index": col, "count": "count", col: col})
    )


def mean_salary_by(df: pd.DataFrame, group_col: str, min_count: int = 5) -> pd.DataFrame:
    """
    Return mean average_salary_lpa grouped by group_col.
    Only include groups with at least min_count rows with salary data.
    """
    sal_df = df.dropna(subset=["average_salary_lpa"])
    agg = (
        sal_df.groupby(group_col)["average_salary_lpa"]
        .agg(["mean", "median", "count"])
        .reset_index()
        .rename(columns={"mean": "avg_salary_lpa", "median": "median_salary_lpa", "count": "n"})
    )
    return agg[agg["n"] >= min_count].sort_values("avg_salary_lpa", ascending=False)


# ---------------------------------------------------------------------------
# Overview page
# ---------------------------------------------------------------------------

def overview_kpis(df: pd.DataFrame) -> dict:
    """Return KPI values for the overview page."""
    sal = df.dropna(subset=["average_salary_lpa"])
    return {
        "total_jobs": len(df),
        "total_companies": df["companyName"].nunique(),
        "total_locations": df["primary_location"].nunique(),
        "avg_salary_lpa": round(sal["average_salary_lpa"].mean(), 2),
        "median_salary_lpa": round(sal["average_salary_lpa"].median(), 2),
    }


def jobs_over_time(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate job postings by days_ago (0 = today, 10 = 10 days ago)."""
    return (
        df.dropna(subset=["days_ago"])
        .groupby("days_ago")
        .size()
        .reset_index(name="job_count")
        .sort_values("days_ago")
    )


# ---------------------------------------------------------------------------
# Demand page
# ---------------------------------------------------------------------------

def demand_by_role(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    return top_n_by_count(df, "title", top_n)


def demand_by_location(df: pd.DataFrame, top_n: int = 20) -> pd.DataFrame:
    return top_n_by_count(df, "primary_location", top_n)


def demand_by_experience(df: pd.DataFrame) -> pd.DataFrame:
    order = ["Fresher (0 Yrs)", "0–2 Yrs", "2–5 Yrs", "5–8 Yrs", "8+ Yrs", "Unknown"]
    counts = (
        df["experience_group"]
        .value_counts()
        .reset_index()
        .rename(columns={"experience_group": "experience_group", "count": "count"})
    )
    counts["sort_key"] = counts["experience_group"].apply(
        lambda x: order.index(x) if x in order else len(order)
    )
    return counts.sort_values("sort_key").drop(columns="sort_key")


# ---------------------------------------------------------------------------
# Salary page
# ---------------------------------------------------------------------------

def salary_distribution(df: pd.DataFrame) -> pd.Series:
    return df["average_salary_lpa"].dropna()


def salary_by_role(df: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    # Limit to top-n roles by total postings for readability
    top_roles = df["title"].value_counts().head(top_n).index.tolist()
    sub = df[df["title"].isin(top_roles)].dropna(subset=["average_salary_lpa"])
    return sub[["title", "average_salary_lpa"]]


def salary_by_location(df: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    top_locs = df["primary_location"].value_counts().head(top_n).index.tolist()
    sub = df[df["primary_location"].isin(top_locs)].dropna(subset=["average_salary_lpa"])
    return sub[["primary_location", "average_salary_lpa"]]


def salary_by_experience(df: pd.DataFrame) -> pd.DataFrame:
    order = ["Fresher (0 Yrs)", "0–2 Yrs", "2–5 Yrs", "5–8 Yrs", "8+ Yrs"]
    sub = df[df["experience_group"].isin(order)].dropna(subset=["average_salary_lpa"])
    return sub[["experience_group", "average_salary_lpa"]]


# ---------------------------------------------------------------------------
# Experience page
# ---------------------------------------------------------------------------

def roles_by_experience(df: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    order = ["Fresher (0 Yrs)", "0–2 Yrs", "2–5 Yrs", "5–8 Yrs", "8+ Yrs"]
    top_roles = df["title"].value_counts().head(top_n).index.tolist()
    sub = df[df["title"].isin(top_roles) & df["experience_group"].isin(order)]
    return (
        sub.groupby(["title", "experience_group"])
        .size()
        .reset_index(name="count")
    )


# ---------------------------------------------------------------------------
# Fresher page
# ---------------------------------------------------------------------------

def fresher_kpis(df: pd.DataFrame) -> dict:
    fr = df[df["is_fresher"]]
    sal = fr.dropna(subset=["average_salary_lpa"])
    return {
        "total_fresher_jobs": len(fr),
        "top_roles": fr["title"].value_counts().head(10).reset_index().rename(
            columns={"title": "role", "count": "count"}),
        "top_cities": fr["primary_location"].value_counts().head(10).reset_index().rename(
            columns={"primary_location": "city", "count": "count"}),
        "top_companies": fr["companyName"].value_counts().head(10).reset_index().rename(
            columns={"companyName": "company", "count": "count"}),
        "avg_salary_lpa": round(sal["average_salary_lpa"].mean(), 2) if len(sal) > 0 else None,
    }


# ---------------------------------------------------------------------------
# Company & Location page
# ---------------------------------------------------------------------------

def top_companies(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    return top_n_by_count(df, "companyName", n)


def top_locations(df: pd.DataFrame, n: int = 20) -> pd.DataFrame:
    return top_n_by_count(df, "primary_location", n)


def avg_salary_by_location(df: pd.DataFrame, top_n: int = 15) -> pd.DataFrame:
    return mean_salary_by(df, "primary_location").head(top_n)


def company_ratings(df: pd.DataFrame, min_reviews: int = 10) -> pd.DataFrame:
    """Return companies with at least min_reviews, sorted by AggregateRating."""
    sub = df.dropna(subset=["AggregateRating"]).copy()
    agg = (
        sub.groupby("companyName")
        .agg(
            avg_rating=("AggregateRating", "mean"),
            total_reviews=("ReviewsCount", "sum"),
            job_postings=("title", "count"),
        )
        .reset_index()
    )
    return (
        agg[agg["total_reviews"] >= min_reviews]
        .sort_values("avg_rating", ascending=False)
        .head(20)
    )


## career_analysis(1).py

In [ ]:
"""
career_analysis.py — Career Skill Gap Analyzer.
Given a target role, identifies the most-requested skills in the dataset
and compares them against a user-supplied skill set.
"""

import pandas as pd
from collections import Counter
from skill_extraction import SKILL_MAP, extract_skills_from_row


# ---------------------------------------------------------------------------
# Supported target roles (mapped from dataset job titles)
# ---------------------------------------------------------------------------

ROLE_KEYWORDS = {
    "Data Analyst": ["data analyst", "data analysis"],
    "Business Analyst": ["business analyst", "business analysis"],
    "Data Scientist": ["data scientist"],
    "Machine Learning Engineer": ["machine learning", "ml engineer"],
    "Data Engineer": ["data engineer"],
    "Software Developer / Engineer": ["software developer", "software engineer", "sde", "swe"],
    "Python Developer": ["python developer", "python engineer"],
    "Java Developer": ["java developer", "java engineer"],
    "Full Stack Developer": ["full stack", "fullstack"],
    "Backend Developer": ["backend developer", "back-end developer"],
    "Frontend Developer": ["frontend developer", "front-end developer", "ui developer"],
    "DevOps Engineer": ["devops", "sre", "site reliability"],
    "Business Development": ["business development"],
    "Sales Executive": ["sales executive", "sales manager"],
    "HR / Recruiter": ["hr executive", "recruiter", "talent acquisition"],
    "Financial Analyst": ["financial analyst", "finance analyst"],
    "Accountant": ["accountant", "accounting"],
    "Project Manager": ["project manager", "program manager"],
}

ALL_CANONICAL_SKILLS = sorted(set(SKILL_MAP.values()))


# ---------------------------------------------------------------------------
# Core functions
# ---------------------------------------------------------------------------

def filter_by_role(df: pd.DataFrame, role_name: str) -> pd.DataFrame:
    """Return rows matching the selected role (case-insensitive keyword match)."""
    keywords = ROLE_KEYWORDS.get(role_name, [role_name.lower()])
    pattern = "|".join(keywords)
    return df[df["title"].str.lower().str.contains(pattern, na=False, regex=True)].copy()


def get_role_skill_demand(role_df: pd.DataFrame, top_n: int = 25) -> pd.DataFrame:
    """
    For a role-filtered DataFrame, return the top-n most frequently
    requested canonical skills with counts and percentage.
    """
    all_skills = [skill for skills in role_df["skills_list"] for skill in skills]
    if not all_skills:
        return pd.DataFrame(columns=["skill", "count", "pct_of_postings"])
    total = len(role_df)
    counts = Counter(all_skills)
    top = pd.DataFrame(counts.most_common(top_n), columns=["skill", "count"])
    top["pct_of_postings"] = (top["count"] / total * 100).round(1)
    return top


def skill_gap_analysis(role_df: pd.DataFrame, user_skills: list[str], top_n: int = 25) -> dict:
    """
    Compare user's current skills against what the market demands for a role.

    Returns a dict with:
      - demand   : top-n skill demand DataFrame (skill, count, pct_of_postings)
      - covered  : skills user already has that are in the top-n
      - gaps     : top-n skills the user doesn't have
      - coverage : % of top-n demand skills the user already has
    """
    demand_df = get_role_skill_demand(role_df, top_n=top_n)
    if demand_df.empty:
        return {"demand": demand_df, "covered": [], "gaps": [], "coverage": 0.0}

    demanded_skills = demand_df["skill"].tolist()
    user_set = set(user_skills)

    covered = [s for s in demanded_skills if s in user_set]
    gaps    = [s for s in demanded_skills if s not in user_set]
    coverage = round(len(covered) / len(demanded_skills) * 100, 1)

    return {
        "demand":   demand_df,
        "covered":  covered,
        "gaps":     gaps,
        "coverage": coverage,
    }


def get_common_skill_combos(role_df: pd.DataFrame, top_n_skills: int = 15, top_combos: int = 10) -> pd.DataFrame:
    """
    Return the most common 2-skill pairs for the role
    (limited to top_n_skills to keep computation fast).
    """
    from itertools import combinations

    top_skills = get_role_skill_demand(role_df, top_n_skills)["skill"].tolist()
    top_set    = set(top_skills)
    co: dict   = {}

    for skills in role_df["skills_list"]:
        filtered = sorted([s for s in skills if s in top_set])
        for a, b in combinations(filtered, 2):
            co[(a, b)] = co.get((a, b), 0) + 1

    if not co:
        return pd.DataFrame(columns=["Skill A", "Skill B", "Co-occurrences"])

    rows = [{"Skill A": a, "Skill B": b, "Co-occurrences": c} for (a, b), c in co.items()]
    return pd.DataFrame(rows).sort_values("Co-occurrences", ascending=False).head(top_combos)


## statistics_analysis(1).py

In [ ]:
"""
statistics_analysis.py — Statistical summaries and comparisons for the
Indian Job Market dataset.  All functions are pure (DataFrame in → result out).
"""

import numpy as np
import pandas as pd
from scipy import stats as scipy_stats


# ---------------------------------------------------------------------------
# Descriptive statistics
# ---------------------------------------------------------------------------

def numeric_summary(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """Full descriptive stat table for a single numeric column."""
    s = df[col].dropna()
    if len(s) == 0:
        return pd.DataFrame({"Statistic": ["No data"], "Value": [None]})

    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1

    rows = [
        ("Count",      len(s)),
        ("Mean",       round(s.mean(), 3)),
        ("Median",     round(s.median(), 3)),
        ("Mode",       round(s.mode().iloc[0], 3) if len(s.mode()) > 0 else None),
        ("Std Dev",    round(s.std(), 3)),
        ("Variance",   round(s.var(), 3)),
        ("Min",        round(s.min(), 3)),
        ("P10",        round(s.quantile(0.10), 3)),
        ("Q1 (P25)",   round(q1, 3)),
        ("Q3 (P75)",   round(q3, 3)),
        ("P90",        round(s.quantile(0.90), 3)),
        ("P95",        round(s.quantile(0.95), 3)),
        ("P99",        round(s.quantile(0.99), 3)),
        ("Max",        round(s.max(), 3)),
        ("IQR",        round(iqr, 3)),
        ("Skewness",   round(s.skew(), 3)),
        ("Kurtosis",   round(s.kurtosis(), 3)),
    ]
    return pd.DataFrame(rows, columns=["Statistic", "Value"])


def outlier_summary(df: pd.DataFrame, col: str) -> dict:
    """Return count and bounds of IQR-based outliers."""
    s = df[col].dropna()
    if len(s) == 0:
        return {}
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - 1.5 * iqr
    hi = q3 + 1.5 * iqr
    outliers = s[(s < lo) | (s > hi)]
    return {
        "lower_fence": round(lo, 2),
        "upper_fence": round(hi, 2),
        "outlier_count": len(outliers),
        "outlier_pct": round(len(outliers) / len(s) * 100, 1),
    }


# ---------------------------------------------------------------------------
# Grouped salary comparisons
# ---------------------------------------------------------------------------

def salary_stats_by_group(df: pd.DataFrame, group_col: str, min_n: int = 10) -> pd.DataFrame:
    """Return mean, median, std, count per group for average_salary_lpa."""
    sal = df.dropna(subset=["average_salary_lpa"])
    agg = (
        sal.groupby(group_col)["average_salary_lpa"]
        .agg(
            Count="count",
            Mean="mean",
            Median="median",
            Std="std",
            Min="min",
            Max="max",
        )
        .reset_index()
        .rename(columns={group_col: "Group"})
    )
    agg = agg[agg["Count"] >= min_n]
    agg[["Mean", "Median", "Std", "Min", "Max"]] = agg[
        ["Mean", "Median", "Std", "Min", "Max"]
    ].round(2)
    return agg.sort_values("Mean", ascending=False)


# ---------------------------------------------------------------------------
# Correlation matrix
# ---------------------------------------------------------------------------

def correlation_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Pearson correlation among key numeric columns."""
    num_cols = [
        "minimumExperience", "maximumExperience", "average_experience",
        "minimumSalary_lpa", "maximumSalary_lpa", "average_salary_lpa",
        "AggregateRating", "ReviewsCount",
    ]
    available = [c for c in num_cols if c in df.columns]
    return df[available].corr(method="pearson").round(3)


# ---------------------------------------------------------------------------
# One-way ANOVA: salary across experience groups
# ---------------------------------------------------------------------------

def anova_salary_by_experience(df: pd.DataFrame) -> dict:
    """
    One-way ANOVA: does average salary differ across experience groups?
    Returns F-stat, p-value, and an interpretation caveat.
    IMPORTANT: ANOVA assumes roughly equal variances and normal distribution
    within groups. With n≈100K, almost any difference will be 'significant'.
    Treat as descriptive evidence, not causal proof.
    """
    exp_order = [
        "Fresher (0 Yrs)", "0\u20132 Yrs", "2\u20135 Yrs",
        "5\u20138 Yrs", "8+ Yrs",
    ]
    groups = []
    for grp in exp_order:
        s = df[df["experience_group"] == grp]["average_salary_lpa"].dropna()
        if len(s) >= 30:
            groups.append(s.values)

    if len(groups) < 2:
        return {"error": "Insufficient data for ANOVA"}

    f_stat, p_val = scipy_stats.f_oneway(*groups)
    return {
        "f_statistic": round(float(f_stat), 3),
        "p_value": float(p_val),
        "significant": p_val < 0.05,
        "caveat": (
            "With ~100K records even tiny differences become statistically significant. "
            "Treat this as descriptive evidence of salary trends, not causal proof."
        ),
    }


# ---------------------------------------------------------------------------
# Percentile table
# ---------------------------------------------------------------------------

def salary_percentiles(df: pd.DataFrame) -> pd.DataFrame:
    """Return salary percentile table at useful breakpoints."""
    s = df["average_salary_lpa"].dropna()
    pts = [5, 10, 20, 25, 30, 40, 50, 60, 70, 75, 80, 90, 95, 99]
    rows = [{"Percentile": f"P{p}", "Salary (LPA)": round(s.quantile(p / 100), 2)} for p in pts]
    return pd.DataFrame(rows)


## sql_analysis(1).py

In [ ]:
"""
sql_analysis.py — Loads the cleaned DataFrame into an in-memory SQLite
database and provides ready-to-run analytical queries.

NOTE: No Streamlit imports here. The page (9_SQL_Analytics.py) applies
@st.cache_resource so this module stays import-safe.
"""

import sqlite3
import statistics
import pandas as pd


class _MedianAgg:
    """SQLite aggregate function that computes the median."""
    def __init__(self):
        self._vals = []
    def step(self, value):
        if value is not None:
            self._vals.append(value)
    def finalize(self):
        if not self._vals:
            return None
        return statistics.median(self._vals)


def build_sqlite_connection(df: pd.DataFrame) -> sqlite3.Connection:
    """
    Create an in-memory SQLite database from the cleaned DataFrame.
    Pure function — no Streamlit dependency. Callers apply caching.
    Registers a custom MEDIAN aggregate function.
    """
    conn = sqlite3.connect(":memory:", check_same_thread=False)
    conn.create_aggregate("MEDIAN", 1, _MedianAgg)

    # Write only the columns we actually need to keep memory low
    cols = [
        "title", "companyName", "primary_location", "experience_group",
        "minimumExperience", "maximumExperience", "average_experience",
        "minimumSalary_lpa", "maximumSalary_lpa", "average_salary_lpa",
        "tagsAndSkills", "AggregateRating", "ReviewsCount",
        "days_ago", "is_fresher", "currency",
    ]
    existing = [c for c in cols if c in df.columns]
    df[existing].to_sql("jobs", conn, if_exists="replace", index=False)
    conn.commit()
    return conn


# ---------------------------------------------------------------------------
# Pre-defined analytical queries
# ---------------------------------------------------------------------------

QUERIES = {
    "1. Top 10 Job Roles by Postings": {
        "sql": """\
SELECT title,
       COUNT(*) AS job_count
FROM   jobs
GROUP  BY title
ORDER  BY job_count DESC
LIMIT  10;""",
        "question": "Which job roles appear most frequently in the dataset?",
    },

    "2. Top 10 Companies by Postings": {
        "sql": """\
SELECT companyName,
       COUNT(*) AS job_count
FROM   jobs
GROUP  BY companyName
ORDER  BY job_count DESC
LIMIT  10;""",
        "question": "Which employers are posting the highest number of jobs?",
    },

    "3. Top 10 Locations by Postings": {
        "sql": """\
SELECT primary_location,
       COUNT(*) AS job_count
FROM   jobs
GROUP  BY primary_location
ORDER  BY job_count DESC
LIMIT  10;""",
        "question": "Which cities have the highest concentration of job postings?",
    },

    "4. Average Salary by Location (Top 15)": {
        "sql": """\
SELECT primary_location,
       ROUND(AVG(average_salary_lpa), 2) AS avg_salary_lpa,
       COUNT(*)                          AS total_jobs,
       COUNT(average_salary_lpa)         AS jobs_with_salary,
       ROUND(MIN(average_salary_lpa), 2) AS min_salary_lpa,
       ROUND(MAX(average_salary_lpa), 2) AS max_salary_lpa
FROM   jobs
WHERE  average_salary_lpa IS NOT NULL
GROUP  BY primary_location
HAVING COUNT(average_salary_lpa) >= 10
ORDER  BY avg_salary_lpa DESC
LIMIT  15;""",
        "question": "Which cities offer the highest average salaries (where salary was disclosed)?",
    },

    "5. Average Salary by Experience Group": {
        "sql": """\
SELECT experience_group,
       ROUND(AVG(average_salary_lpa), 2) AS avg_salary_lpa,
       COUNT(*)                          AS total_jobs,
       COUNT(average_salary_lpa)         AS jobs_with_salary,
       ROUND(MIN(average_salary_lpa), 2) AS min_salary_lpa,
       ROUND(MAX(average_salary_lpa), 2) AS max_salary_lpa
FROM   jobs
WHERE  experience_group NOT IN ('Unknown')
  AND  average_salary_lpa IS NOT NULL
GROUP  BY experience_group
ORDER  BY avg_salary_lpa DESC;""",
        "question": "How does average salary vary across experience levels?",
    },

    "6. Jobs Requiring Python": {
        "sql": """\
SELECT title,
       companyName,
       primary_location,
       experience_group,
       average_salary_lpa
FROM   jobs
WHERE  LOWER(tagsAndSkills) LIKE '%python%'
ORDER  BY average_salary_lpa DESC NULLS LAST
LIMIT  50;""",
        "question": "Which Python job openings are available and what salaries do they offer?",
    },

    "7. Jobs Requiring SQL": {
        "sql": """\
SELECT title,
       companyName,
       primary_location,
       experience_group,
       average_salary_lpa
FROM   jobs
WHERE  LOWER(tagsAndSkills) LIKE '%sql%'
ORDER  BY average_salary_lpa DESC NULLS LAST
LIMIT  50;""",
        "question": "Which SQL-related job openings are available?",
    },

    "8. Jobs Mentioning Both Python & SQL": {
        "sql": """\
SELECT title,
       companyName,
       primary_location,
       experience_group,
       average_salary_lpa
FROM   jobs
WHERE  LOWER(tagsAndSkills) LIKE '%python%'
  AND  LOWER(tagsAndSkills) LIKE '%sql%'
ORDER  BY average_salary_lpa DESC NULLS LAST
LIMIT  50;""",
        "question": "Which roles require both Python and SQL — the core data analyst skill stack?",
    },

    "9. Top Companies for Fresher Roles": {
        "sql": """\
SELECT companyName,
       COUNT(*)                          AS fresher_jobs,
       ROUND(AVG(average_salary_lpa), 2) AS avg_salary_lpa
FROM   jobs
WHERE  is_fresher = 1
GROUP  BY companyName
HAVING fresher_jobs >= 5
ORDER  BY fresher_jobs DESC
LIMIT  20;""",
        "question": "Which companies hire the most entry-level/fresher candidates?",
    },

    "10. Fresher Jobs by Location": {
        "sql": """\
SELECT primary_location,
       COUNT(*) AS fresher_jobs
FROM   jobs
WHERE  is_fresher = 1
GROUP  BY primary_location
ORDER  BY fresher_jobs DESC
LIMIT  15;""",
        "question": "Which cities have the most fresher/entry-level opportunities?",
    },

    "11. Salary Comparison Across Experience Groups (CTE)": {
        "sql": """\
WITH salary_stats AS (
    SELECT experience_group,
           COUNT(*)                          AS total_jobs,
           COUNT(average_salary_lpa)         AS salary_disclosed,
           ROUND(AVG(average_salary_lpa), 2) AS avg_lpa,
           ROUND(MIN(average_salary_lpa), 2) AS min_lpa,
           ROUND(MAX(average_salary_lpa), 2) AS max_lpa
    FROM   jobs
    WHERE  experience_group NOT IN ('Unknown')
    GROUP  BY experience_group
)
SELECT *,
       ROUND(CAST(salary_disclosed AS FLOAT) / total_jobs * 100, 1) AS pct_disclosed
FROM   salary_stats
ORDER  BY avg_lpa DESC;""",
        "question": "Comprehensive salary statistics by experience level including disclosure rates.",
    },

    "12. Top Roles With Highest Average Salary": {
        "sql": """\
WITH role_stats AS (
    SELECT title,
           COUNT(*)                          AS total_jobs,
           COUNT(average_salary_lpa)         AS salary_count,
           ROUND(AVG(average_salary_lpa), 2) AS avg_salary_lpa,
           ROUND(MIN(average_salary_lpa), 2) AS min_salary_lpa,
           ROUND(MAX(average_salary_lpa), 2) AS max_salary_lpa
    FROM   jobs
    WHERE  average_salary_lpa IS NOT NULL
    GROUP  BY title
    HAVING salary_count >= 10
)
SELECT *
FROM   role_stats
ORDER  BY avg_salary_lpa DESC
LIMIT  20;""",
        "question": "Which job roles command the highest average salaries (minimum 10 postings with salary)?",
    },
}


def run_query(conn: sqlite3.Connection, sql: str) -> pd.DataFrame:
    """Execute a SQL query and return results as a DataFrame."""
    try:
        return pd.read_sql_query(sql, conn)
    except Exception as e:
        return pd.DataFrame({"Error": [str(e)]})


## ml_salary_prediction(1).py

In [ ]:
"""
ml_salary_prediction.py — Salary prediction pipeline for the Indian
Job Market dataset.

Target : average_salary_lpa
Features: minimumExperience, maximumExperience, title (top-N), primary_location (top-N)
Models  : Linear Regression (baseline), Random Forest, Gradient Boosting

NOTE: No Streamlit imports here. The page (11_Salary_Prediction.py) applies
@st.cache_data so this module stays import-safe.
"""

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

TARGET = "average_salary_lpa"
TOP_N_ROLES = 50
TOP_N_LOCS  = 50
RARE_LABEL  = "__other__"

MODELS = {
    "Linear Regression (Baseline)": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}


# ---------------------------------------------------------------------------
# Feature preparation
# ---------------------------------------------------------------------------

def _prepare_features(df: pd.DataFrame):
    """
    Return X (feature DataFrame) and y (target Series) after
    - filtering to rows with salary disclosed
    - reducing high-cardinality categoricals to top-N + RARE_LABEL
    """
    sal = df.dropna(subset=[TARGET]).copy()
    if len(sal) < 200:
        raise ValueError("Insufficient salary data to train a model (need ≥ 200 rows).")

    # Reduce cardinality
    for col, top_n in [("title", TOP_N_ROLES), ("primary_location", TOP_N_LOCS)]:
        top_vals = sal[col].value_counts().head(top_n).index
        sal[col] = sal[col].where(sal[col].isin(top_vals), other=RARE_LABEL)

    feature_cols = ["minimumExperience", "maximumExperience", "title", "primary_location"]
    X = sal[feature_cols].copy()
    y = sal[TARGET]
    return X, y, sal[feature_cols + [TARGET]]


def _build_pipeline(model) -> Pipeline:
    numeric_features  = ["minimumExperience", "maximumExperience"]
    categoric_features = ["title", "primary_location"]

    numeric_transformer = SimpleImputer(strategy="median")
    categoric_transformer = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value=RARE_LABEL)),
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categoric_transformer, categoric_features),
    ])

    return Pipeline([("preprocessor", preprocessor), ("model", model)])


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

def train_models(df: pd.DataFrame) -> dict:
    """
    Train all models and return a results dict.
    Pure function — no Streamlit dependency. Callers apply @st.cache_data.
    """
    try:
        X, y, _ = _prepare_features(df)
    except ValueError as e:
        return {"error": str(e)}

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    results = {}
    for name, model in MODELS.items():
        pipe = _build_pipeline(model)
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        mae  = mean_absolute_error(y_test, y_pred)
        rmse = mean_squared_error(y_test, y_pred) ** 0.5
        r2   = r2_score(y_test, y_pred)

        results[name] = {
            "pipeline": pipe,
            "mae":  round(mae,  3),
            "rmse": round(rmse, 3),
            "r2":   round(r2,   3),
            "y_test":  y_test.values,
            "y_pred":  y_pred,
        }

    return results


# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------

def predict_salary(pipeline, title: str, location: str,
                   min_exp: float, max_exp: float) -> float:
    """Return a salary estimate in LPA for one job profile."""
    row = pd.DataFrame([{
        "title": title,
        "primary_location": location,
        "minimumExperience": min_exp,
        "maximumExperience": max_exp,
    }])
    return float(pipeline.predict(row)[0])


# ---------------------------------------------------------------------------
# Helpers for UI dropdowns
# ---------------------------------------------------------------------------

def get_top_roles(df: pd.DataFrame, n: int = TOP_N_ROLES) -> list:
    return sorted(df["title"].value_counts().head(n).index.tolist())


def get_top_locations(df: pd.DataFrame, n: int = TOP_N_LOCS) -> list:
    return sorted(df["primary_location"].value_counts().head(n).index.tolist())


## app(1).py

In [ ]:
"""
app.py — Entry point for the Indian Job Market Intelligence dashboard.

Run with:  streamlit run app.py
"""

import os
import sys

import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

# Make src/ importable from anywhere
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "src"))

from data_loader import load_raw_data, get_dataset_info, DEFAULT_DATA_PATH
from data_cleaning import clean_data
from feature_engineering import engineer_features
from skill_extraction import add_skills_list
from analysis import overview_kpis, jobs_over_time, demand_by_experience
from ui_helpers import (
    inject_css, render_sidebar,
    page_header, section, kpi_row, divider,
    insight_card, apply_chart_style,
    QUAL_COLORS, PRIMARY, ACCENT_GREEN,
    SURFACE, TEXT, MUTED, BORDER,
    LOGO_SVG_LARGE,
)

# ── Page config ────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Indian Job Market Intelligence",
    page_icon="🇮🇳",
    layout="wide",
    initial_sidebar_state="expanded",
)

inject_css()

# ── Load & cache ───────────────────────────────────────────────────────────
@st.cache_data(show_spinner=False)
def _cached_raw(path: str):
    return load_raw_data(path)


@st.cache_data(show_spinner=False)
def get_clean_data(path: str):
    raw = load_raw_data(path)
    cleaned = clean_data(raw)
    featured = engineer_features(cleaned)
    featured = add_skills_list(featured)
    return featured


# ── Shared sidebar ─────────────────────────────────────────────────────────
data_path = render_sidebar(DEFAULT_DATA_PATH)

# ── Load data ──────────────────────────────────────────────────────────────
with st.spinner("Loading & cleaning dataset…"):
    try:
        df = get_clean_data(data_path)
        st.session_state["df"] = df
    except FileNotFoundError as e:
        st.error(str(e))
        st.stop()
    except Exception as e:
        st.error(f"Unexpected error loading data: {e}")
        st.stop()

# ── Hero ───────────────────────────────────────────────────────────────────
hero_col, logo_col = st.columns([4, 1])
with hero_col:
    st.markdown(
        """
        <div style="padding:1.6rem 0 0.8rem 0;">
            <div style="font-size:0.75rem;color:#4f8ef7;font-weight:600;
                        letter-spacing:0.12em;text-transform:uppercase;margin-bottom:0.55rem;">
                IJMI &nbsp;·&nbsp; ANALYTICS PLATFORM &nbsp;·&nbsp; INDIA
            </div>
            <div style="font-size:2.3rem;font-weight:800;color:#f8fafc;line-height:1.15;
                        margin-bottom:0.6rem;letter-spacing:-0.01em;">
                Indian Job Market<br>
                <span style="color:#38bdf8;">Intelligence</span>
            </div>
            <div style="font-size:0.95rem;color:#94a3b8;max-width:560px;line-height:1.65;">
                A professional analytics platform built on&nbsp;<strong style="color:#cbd5e1;">97,682</strong>
                real job postings from Naukri.com. Explore demand trends, salary patterns,
                skill gaps, and career insights&nbsp;&#8212; all in one place.
            </div>
        </div>
        """,
        unsafe_allow_html=True,
    )

with logo_col:
    st.markdown(LOGO_SVG_LARGE, unsafe_allow_html=True)

# ── KPI strip ─────────────────────────────────────────────────────────────
kpis = overview_kpis(df)
avg_sal = float(kpis["avg_salary_lpa"]) if kpis.get("avg_salary_lpa") is not None else 0.0

kpi_row([
    ("Total Job Postings",    f"{kpis['total_jobs']:,}",      "Cleaned & deduplicated"),
    ("Unique Companies",      f"{kpis['total_companies']:,}", "Hiring across India"),
    ("Locations Covered",     f"{kpis['total_locations']:,}", "Cities & regions"),
    ("Avg Advertised Salary", f"\u20b9{avg_sal:.1f} LPA",     "Where salary disclosed"),
])

divider()

# ── Two-column charts ──────────────────────────────────────────────────────
col_left, col_right = st.columns(2, gap="large")

with col_left:
    section("Job Posting Activity")
    jot = jobs_over_time(df)
    if not jot.empty:
        fig_trend = px.area(
            jot,
            x="days_ago",
            y="job_count",
            color_discrete_sequence=[PRIMARY],
            labels={"days_ago": "Days Ago", "job_count": "Postings"},
        )
        fig_trend.update_traces(
            line_color=PRIMARY,
            fillcolor="rgba(79,142,247,0.14)",
            hovertemplate="<b>%{x} days ago</b><br>%{y:,} postings<extra></extra>",
        )
        fig_trend.update_xaxes(autorange="reversed")
        fig_trend.update_layout(title="How recent are the job postings?")
        st.plotly_chart(apply_chart_style(fig_trend, 295), use_container_width=True)

with col_right:
    section("Experience Distribution")
    exp_df = demand_by_experience(df)
    exp_df = exp_df[exp_df["experience_group"] != "Unknown"]
    if not exp_df.empty:
        fig_exp = px.pie(
            exp_df,
            names="experience_group",
            values="count",
            color_discrete_sequence=QUAL_COLORS,
            hole=0.46,
        )
        fig_exp.update_traces(
            textfont_size=11,
            textfont_color="#e8eaf0",
            marker=dict(line=dict(color="#0f1117", width=2)),
            hovertemplate="<b>%{label}</b><br>%{value:,} postings (%{percent})<extra></extra>",
        )
        fig_exp.update_layout(title="How are jobs distributed by seniority?")
        st.plotly_chart(apply_chart_style(fig_exp, 295), use_container_width=True)

divider()

# ── Three-column mini charts ───────────────────────────────────────────────
section("Quick Snapshot")
c1, c2, c3 = st.columns(3, gap="large")

with c1:
    st.markdown(
        "<div style='font-size:0.78rem;font-weight:600;color:#8b92a8;"
        "text-transform:uppercase;letter-spacing:0.07em;margin-bottom:6px;'>Top Roles</div>",
        unsafe_allow_html=True,
    )
    top_roles = df["title"].value_counts().head(8).reset_index()
    top_roles.columns = ["title", "count"]
    fig_r = px.bar(
        top_roles.sort_values("count"),
        x="count", y="title",
        orientation="h",
        color_discrete_sequence=[PRIMARY],
        labels={"count": "Postings", "title": ""},
    )
    fig_r.update_traces(marker_line_width=0,
                        hovertemplate="<b>%{y}</b><br>%{x:,}<extra></extra>")
    st.plotly_chart(apply_chart_style(fig_r, 310), use_container_width=True)

with c2:
    st.markdown(
        "<div style='font-size:0.78rem;font-weight:600;color:#8b92a8;"
        "text-transform:uppercase;letter-spacing:0.07em;margin-bottom:6px;'>Top Locations</div>",
        unsafe_allow_html=True,
    )
    top_locs = df["primary_location"].value_counts().head(8).reset_index()
    top_locs.columns = ["primary_location", "count"]
    fig_l = px.bar(
        top_locs.sort_values("count"),
        x="count", y="primary_location",
        orientation="h",
        color_discrete_sequence=[ACCENT_GREEN],
        labels={"count": "Postings", "primary_location": ""},
    )
    fig_l.update_traces(marker_line_width=0,
                        hovertemplate="<b>%{y}</b><br>%{x:,}<extra></extra>")
    st.plotly_chart(apply_chart_style(fig_l, 310), use_container_width=True)

with c3:
    st.markdown(
        "<div style='font-size:0.78rem;font-weight:600;color:#8b92a8;"
        "text-transform:uppercase;letter-spacing:0.07em;margin-bottom:6px;'>Top Skills</div>",
        unsafe_allow_html=True,
    )
    from collections import Counter
    skill_counts = Counter(
        skill.strip()
        for skills in df["skills_list"].dropna()
        for skill in skills
        if skill.strip()
    )
    top_skills_df = pd.DataFrame(skill_counts.most_common(8), columns=["skill", "count"])
    top_skills_df = top_skills_df.sort_values("count")
    fig_s = px.bar(
        top_skills_df,
        x="count", y="skill",
        orientation="h",
        color_discrete_sequence=["#f59e0b"],
        labels={"count": "Mentions", "skill": ""},
    )
    fig_s.update_traces(marker_line_width=0,
                        hovertemplate="<b>%{y}</b><br>%{x:,}<extra></extra>")
    st.plotly_chart(apply_chart_style(fig_s, 310), use_container_width=True)

divider()

# ── Key Insights ───────────────────────────────────────────────────────────
section("Key Insights")

top_role_name  = df["title"].value_counts().index[0]           if not df.empty else "\u2014"
top_role_count = int(df["title"].value_counts().iloc[0])       if not df.empty else 0
top_city       = df["primary_location"].value_counts().index[0] if not df.empty else "\u2014"
top_city_count = int(df["primary_location"].value_counts().iloc[0]) if not df.empty else 0
fresher_pct    = round(df["is_fresher"].mean() * 100, 1) if "is_fresher" in df.columns else 0
sal_coverage   = round(df["average_salary_lpa"].notna().mean() * 100, 1)

insight_card(
    f"<b>{top_role_name}</b> is the most-posted role with "
    f"<b>{top_role_count:,}</b> listings, accounting for "
    f"{top_role_count / kpis['total_jobs'] * 100:.1f}% of all postings in the dataset.",
    "\U0001f4cc",
)
insight_card(
    f"<b>{top_city}</b> leads hiring with <b>{top_city_count:,}</b> postings, "
    f"making it the #1 city by job volume in the dataset.",
    "\U0001f3d9\ufe0f",
)
insight_card(
    f"<b>{fresher_pct}%</b> of postings require 0 years of minimum experience, "
    f"indicating a meaningful share of entry-level roles in the dataset.",
    "\U0001f331",
)
insight_card(
    f"Salary data is available for <b>{sal_coverage}%</b> of postings. "
    f"The average advertised salary is <b>\u20b9{avg_sal:.1f} LPA</b> across disclosed roles.",
    "\U0001f4b0",
)

divider()

# ── Navigation grid ────────────────────────────────────────────────────────
section("Explore the Dashboard")

pages = [
    ("📊", "Job Market Overview",    "pages/1_Job_Market_Overview.py",  "Jobs, top roles, companies & trends"),
    ("🔍", "Job Demand Analysis",    "pages/2_Job_Demand.py",           "Demanded roles, cities & experience"),
    ("\U0001f9e0", "Skill Intelligence", "pages/3_Skill_Intelligence.py", "Top skills, gaps & co-occurrence"),
    ("💰", "Salary Intelligence",    "pages/4_Salary_Intelligence.py",  "Salary distributions by role & city"),
    ("📈", "Experience Analysis",    "pages/5_Experience_Analysis.py",  "Jobs by seniority & salary trends"),
    ("🌱", "Fresher Opportunities",  "pages/6_Fresher_Jobs.py",         "Entry-level jobs & top companies"),
    ("🏢", "Company & Location",     "pages/7_Company_Location.py",     "Top employers, cities & ratings"),
    ("🔎", "Job Explorer",           "pages/13_Job_Explorer.py",        "Search & filter 97K+ postings"),
    ("\U0001f5c4\ufe0f", "SQL Analytics", "pages/9_SQL_Analytics.py",   "Analytical SQL on in-memory SQLite"),
    ("📐", "Statistical Insights",   "pages/10_Statistical_Insights.py","Descriptive stats & correlations"),
    ("🤖", "Salary Prediction",      "pages/11_Salary_Prediction.py",   "ML model to estimate salary"),
    ("🎯", "Career Skill Gap",       "pages/12_Career_Skill_Gap.py",    "Your skills vs employer demand"),
    ("📁", "Dataset & Quality",      "pages/8_Dataset_Quality.py",      "Data quality & cleaning report"),
]

nav_cols = st.columns(4, gap="small")
for i, (icon, title, page_path, desc) in enumerate(pages):
    nav_cols[i % 4].page_link(
        page_path,
        label=f"{icon} **{title}**",
        help=desc,
    )

# ── Footer ─────────────────────────────────────────────────────────────────
st.markdown(
    "<div class='dash-footer'>"
    "<strong style='color:#c4c9d8;'>Indian Job Market Intelligence</strong>"
    "&nbsp;&nbsp;·&nbsp;&nbsp;"
    "Data Analytics Portfolio Project"
    "<br>"
    "Python &nbsp;\u00b7&nbsp; Pandas &nbsp;\u00b7&nbsp; SQL &nbsp;\u00b7&nbsp; "
    "Plotly &nbsp;\u00b7&nbsp; Streamlit &nbsp;\u00b7&nbsp; Scikit-learn"
    "<br>"
    "<span style='opacity:0.6;'>"
    "Dataset represents collected job postings and should not be interpreted "
    "as a complete representation of the entire Indian job market."
    "</span>"
    "</div>",
    unsafe_allow_html=True,
)
